# Exploratory Data Analysis - Electricity Theft Detection

This notebook explores the synthetic electricity consumption dataset used in this project.
The goal is to understand consumption patterns, spot differences between normal and
suspicious behavior, and justify the feature-engineering choices made later in
`src/feature_engineering.py`.

**Note:** The dataset is synthetic (see `src/generate_dataset.py` for how and why it was
generated). The `Theft_Flag_GroundTruth` column is used here only for exploratory
comparison - it is NOT used to train the unsupervised models.

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_preprocessing import clean_dataset, outlier_summary
from feature_engineering import engineer_features, ENGINEERED_FEATURES

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

pd.set_option("display.max_columns", None)

## 1. Load and Clean Data

In [ ]:
df = clean_dataset("../data/electricity_data.csv")
df = engineer_features(df)
df.shape

In [ ]:
df.head()

In [ ]:
df.describe().T

## 2. Missing Values & Outliers (before/after preprocessing)

`clean_dataset()` already handled missing values. Here we just double check, and look at
outliers detected using the IQR method (outliers are flagged for awareness, not
automatically removed - they may be exactly the records we care about).

In [ ]:
print("Remaining missing values:")
print(df.isna().sum()[df.isna().sum() > 0])

outlier_summary(df)

## 3. Distribution of Electricity Consumption

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df["Electricity_Consumption_kWh"], bins=50, kde=True, ax=ax)
ax.set_title("Distribution of Monthly Electricity Consumption")
ax.set_xlabel("Consumption (kWh)")
plt.show()

## 4. Monthly Consumption Trends

In [ ]:
monthly_avg = df.groupby("Month")["Electricity_Consumption_kWh"].mean()

fig, ax = plt.subplots()
monthly_avg.plot(marker="o", ax=ax)
ax.set_title("Average Consumption by Month (all customers)")
ax.set_xlabel("Month")
ax.set_ylabel("Average Consumption (kWh)")
plt.show()

## 5. Normal vs Suspicious Consumption Patterns

Comparing the distribution of consumption change (%) for records with a ground-truth
theft flag of 0 vs 1. This justifies using `Consumption_Change_Pct` as a key engineered
feature - suspicious records skew noticeably toward large negative changes, though there
is still overlap (which is realistic - not every large drop is theft).

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=df, x="Theft_Flag_GroundTruth", y="Consumption_Change_Pct", ax=ax)
ax.set_xticklabels(["Normal (0)", "Suspicious/Theft - ground truth (1)"])
ax.set_title("Consumption Change % : Normal vs Suspicious (ground truth)")
plt.show()

## 6. Consumption by Customer Type

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=df, x="Customer_Type", y="Electricity_Consumption_kWh", ax=ax)
ax.set_title("Consumption by Customer Type")
plt.show()

## 7. Peak vs Off-Peak Consumption

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["Off_Peak_Consumption"], df["Peak_Consumption"], alpha=0.3, s=10)
ax.set_xlabel("Off-Peak Consumption (kWh)")
ax.set_ylabel("Peak Consumption (kWh)")
ax.set_title("Peak vs Off-Peak Consumption")
plt.show()

## 8. Correlation Heatmap

In [ ]:
numeric_cols = ENGINEERED_FEATURES + ["Theft_Flag_GroundTruth"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap of Engineered Features")
plt.show()

## 9. Boxplots for Important Numerical Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(y=df["Power_Factor"], ax=axes[0])
axes[0].set_title("Power Factor")
sns.boxplot(y=df["Consumption_Variability"], ax=axes[1])
axes[1].set_title("Consumption Variability")
sns.boxplot(y=df["Peak_OffPeak_Ratio"], ax=axes[2])
axes[2].set_title("Peak / Off-Peak Ratio")
plt.tight_layout()
plt.show()

## 10. Anomaly Score Distribution

This cell requires the models to already be trained (`python src/train_model.py`), since
it loads the scored dataset. It shows how anomaly scores are distributed across all
customers - most cluster at low scores (normal), with a smaller tail of higher scores.

In [ ]:
import os
scored_path = "../data/scored_customers.csv"
if os.path.exists(scored_path):
    scored_df = pd.read_csv(scored_path)
    fig, ax = plt.subplots()
    sns.histplot(scored_df["Anomaly_Score"], bins=50, kde=True, ax=ax)
    ax.set_title("Anomaly Score Distribution (Isolation Forest)")
    ax.set_xlabel("Anomaly Score (0 = normal, 1 = most anomalous)")
    plt.show()
else:
    print("Run 'python src/train_model.py' first to generate the scored dataset.")

## 11. Risk-Category Distribution

The Normal / Suspicious / High Risk labels are rule-based conversions of the anomaly
score (see `assign_risk_level()` in `src/train_model.py`) - not something the model
decides on its own. This chart shows how many customer-month RECORDS fall into each
category (record-level, not unique customers - see README "Customer-Level vs
Record-Level Analysis" for why that distinction matters).

In [ ]:
if os.path.exists(scored_path):
    order = ["Normal", "Suspicious", "High Risk"]
    colors = ["#4C9A6A", "#E8B84B", "#D1495B"]
    fig, ax = plt.subplots()
    sns.countplot(data=scored_df, x="Risk_Level", order=order, hue="Risk_Level", palette=colors, legend=False, ax=ax)
    ax.set_title("Risk Level Distribution (record-level, all months)")
    ax.set_xlabel("")
    ax.set_ylabel("Number of Customer-Month Records")
    plt.show()
else:
    print("Run 'python src/train_model.py' first to generate the scored dataset.")

## 12. Example Customer Consumption History

A single customer's actual monthly consumption plotted against their own historical
baseline (`Average_Consumption_6_Months`, which excludes the current month - see
`src/generate_dataset.py`). This is the same idea shown per-customer in the Streamlit
dashboard's "Customer Analysis" tab - useful here to sanity-check the feature by eye.

In [ ]:
if os.path.exists(scored_path):
    # pick a customer who was flagged Suspicious or High Risk at some point, if one exists
    flagged_ids = scored_df.loc[scored_df["Risk_Level"] != "Normal", "Customer_ID"].unique()
    example_id = flagged_ids[0] if len(flagged_ids) > 0 else scored_df["Customer_ID"].iloc[0]

    example = scored_df[scored_df["Customer_ID"] == example_id].sort_values("Month")

    fig, ax = plt.subplots()
    ax.plot(example["Month"], example["Electricity_Consumption_kWh"], marker="o", label="Actual Consumption")
    ax.plot(example["Month"], example["Average_Consumption_6_Months"], marker="x", linestyle="--",
            label="Historical Baseline (previous months only)")
    ax.set_title(f"Consumption History - Customer {example_id}")
    ax.set_xlabel("Month")
    ax.set_ylabel("Consumption (kWh)")
    ax.legend()
    plt.show()

    print(example[["Month", "Electricity_Consumption_kWh", "Average_Consumption_6_Months",
                    "Consumption_Change_Pct", "Anomaly_Score", "Risk_Level"]].to_string(index=False))
else:
    print("Run 'python src/train_model.py' first to generate the scored dataset.")

## Summary of EDA Findings

- Consumption is right-skewed, as expected for utility data, with Commercial and
  Industrial customers consuming far more than Residential customers.
- Suspicious (ground-truth) records show a visible tendency toward large negative
  `Consumption_Change_Pct`, supporting its use as an engineered feature - though the
  overlap with normal records confirms this alone isn't a reliable rule.
- Power Factor shows a small cluster of unusually low values, consistent with the
  simulated "tampering" pattern in the synthetic data generator.
- Risk categories are heavily imbalanced toward "Normal", as expected for a rare-event
  detection problem - this is why the project evaluates with precision/recall/F1/PR-AUC
  rather than accuracy alone (see `src/evaluate_model.py`).
- These observations directly motivated the feature-engineering choices in
  `src/feature_engineering.py`.